# V2 — Tối ưu S4 Matching bằng shared-memory descriptor tiling

**Phụ trách: Khánh.** V2 kế thừa V1, chỉ thay cách đọc candidate descriptor trong S4. S1–S3, S5–S6, thuật toán và tham số giữ nguyên. Mục tiêu là giảm thời gian matching và đánh giá tác động lên toàn pipeline.

Phần V2 được viết theo đúng năm mục của đề: **a. Bottleneck; b. Idea; c. Implementation; d. Result; e. Analysis**. V0 là reference correctness/speedup; V1 là bản đối chiếu trực tiếp của tối ưu mới.

Notebook chưa có số đo thực tế. Chạy trên **Colab GPU → Runtime → Run all**, rồi lưu toàn bộ output. Mọi nhận xét có số liệu phải lấy từ các cell đã chạy; không tự điền kết quả dự kiến thành kết quả đo.

## Setup và build

Source nằm trong `src/`, công cụ riêng V2 nằm trong `src/tools/v2/`. Notebook chỉ build/run source và trình bày kết quả. Đưa source mới nhất lên nhánh `v2` trước khi clone, hoặc upload đầy đủ source và bật `USE_EXISTING_SOURCE`.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, shlex, subprocess, json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import display, Markdown, HTML, Code

REPO_URL = 'https://github.com/nganbee/panorama-stitching-project.git'
BRANCH = 'v2'
REPO_DIR = Path('/content/panorama-stitching-project')
USE_EXISTING_SOURCE = False

def run(command, check=True):
    args = shlex.split(command) if isinstance(command,str) else [str(x) for x in command]
    print('$',shlex.join(args),flush=True)
    with subprocess.Popen(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        for line in process.stdout:
            print(line,end='',flush=True)
        status = process.wait()
    if check and status:
        raise RuntimeError(f'Command failed with exit code {status}; inspect its output.')
    return status

if USE_EXISTING_SOURCE:
    assert REPO_DIR.is_dir(), 'Upload source and set REPO_DIR first.'
elif not REPO_DIR.exists():
    run(['git','clone','--branch',BRANCH,REPO_URL,REPO_DIR])
else:
    run(['git','-C',REPO_DIR,'fetch','origin',BRANCH])
    run(['git','-C',REPO_DIR,'checkout',BRANCH])
    run(['git','-C',REPO_DIR,'pull','--ff-only','origin',BRANCH])
os.chdir(REPO_DIR)
run('nvidia-smi')
run('nvcc --version')
run('g++ --version')
run('git rev-parse HEAD',check=False)

sites = ['location1','location2','location3','challenge']
sizes = {'small_720p':(1280,720),'medium_1080p':(1920,1080),'large_4k':(3840,2160)}
for site in sites:
    for size,expected in sizes.items():
        for name in ['1.jpg','2.jpg']:
            path = Path('dataset')/site/size/name
            assert path.is_file(), f'Missing input: {path}'
            with PILImage.open(path) as image:
                assert image.size == expected, (str(path),image.size,expected)
print('Dataset ready:',len(sites)*len(sizes),'cases')

In [ ]:
run('bash src/tools/v2/build.sh')
run('build/stitch_v2 --list-versions')
params = json.loads(subprocess.check_output(['build/stitch_v2','--dump-params'],text=True))
display(pd.DataFrame([params]))
assert all(v in params['available_versions'] for v in [0,1,2])
# Check the changed stage and basic integration before the full benchmark.
run('build/stage34_v2_selftest')
run('build/test_v2_pipeline')

## a. Bottleneck — vì sao chọn S4?

Đo V0 và V1 trên cùng bộ ảnh để có baseline trong **chính phiên Colab GPU này**. Dùng timing từng stage của V1 để xác định S4 có đáng tối ưu; không suy bottleneck GPU từ thời gian CPU V0. Bảng dưới giữ S1–S6, tổng kernel, H2D, D2H và end-to-end để không bỏ qua chi phí truyền dữ liệu.

**Phương pháp đo dùng chung:** mỗi case/version chạy một warm-up, sau đó năm lần đo và báo cáo trung bình. GPU stage/copy dùng CUDA events trong pipeline; end-to-end bao gồm allocation, launch, đồng bộ và truyền dữ liệu, loại đọc/ghi ảnh. Lưu từng lần đo và minimum để xem thêm khi dao động lớn. Không cộng kernel/copy vào end-to-end lần nữa.

Dataset gồm ba cặp chuẩn và một cặp khó ở ba độ phân giải. Small phục vụ kiểm tra, Medium là benchmark chính, Large cho thấy khả năng mở rộng theo kích thước. Các version dùng cùng ảnh, tham số và cấu hình build. Giữ nguyên GPU/runtime trong suốt notebook; nếu đổi runtime, chạy lại từ đầu để không trộn số liệu.

In [ ]:
RUN_DIR = Path('reports/v2_benchmark')/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
run([sys.executable,'src/tools/v2/benchmark.py','--phase','baseline','--run-dir',RUN_DIR])
baseline = pd.read_csv(RUN_DIR/'summary.csv')
columns = ['case']+[f'v1_S{i}_ms' for i in range(1,7)]+[
    'v1_kernel_total_ms','v1_h2d_ms','v1_d2h_ms','v1_end_to_end_ms','v1_S4_kernel_share']
display(baseline.reindex(columns=columns))
print('Units: ms; S4 share is relative to GPU kernel_total, not end-to-end.')

### Bằng chứng profiling V1

Đề yêu cầu output profiler hỗ trợ lý do chọn tối ưu. Script chọn case có S4 V1 lâu nhất từ baseline, thu Nsight Systems hoặc Nsight Compute, lưu raw report cùng text/CSV. Đối chiếu kernel `nearest_kernel` với bảng thời gian để giải thích bottleneck. Việc có report chưa tự chứng minh S4 là stage chi phối; phải đọc kết quả trước khi chốt nhận xét.

Nếu môi trường thiếu profiler hoặc bị chặn quyền thu dữ liệu, ghi rõ phần bằng chứng này chưa có. Vẫn giữ số đo benchmark, nhưng chưa kết luận đã đáp ứng đầy đủ yêu cầu profiling. Thời gian chạy dưới profiler **không** được dùng làm timing/speedup chính.

In [ ]:
run([sys.executable,'src/tools/v2/profile.py','--run-dir',RUN_DIR,'--version','1'])
profile_v1 = json.loads((RUN_DIR/'profiling/profile_v1.json').read_text(encoding='utf-8'))
print('Profiling:',profile_v1['status'])
if profile_v1['status']=='collected':
    print('Case:',profile_v1['case'],'| tool:',profile_v1['tool'])
    print('Report:',profile_v1['report'])
else:
    print('Profiling evidence is missing; retain actual logs and collect it in a supported environment.')

## b. Idea — tái sử dụng candidate descriptor bằng shared memory

Trong S4 V1, một thread xử lý một query và quét candidate descriptor từ global memory. Nhiều query cùng so với một tập candidate. V2 giữ cách chia query, nhưng cho block hợp tác tải từng tile candidate vào shared memory để các thread dùng chung.

Ý tưởng kỳ vọng giảm đọc global memory lặp lại. Lợi ích thực tế phụ thuộc cache V1, số descriptor và chi phí đồng bộ, nên không bảo đảm mọi case đều nhanh hơn. V2 chỉ thử **descriptor tiling**; không gộp thay đổi precision, descriptor layout, warp/query hoặc GPU data residency vào cùng version.

## c. Implementation — phần thay đổi so với V1

```text
S1 preprocessing [V1] → S2 keypoints [V1] → S3 BRIEF [V1]
 → S4 descriptor tiling [V2]
 → S5 homography [V1] → S6 warp/blend/crop [V1] → panorama
```

- **Mapping:** 128 thread/block, một query/thread; tile chứa 128 descriptor 256 bit, dùng 4 KB shared memory.
- **Tải và sử dụng:** các thread nạp candidate, đồng bộ, quét tile theo index tăng dần, rồi đồng bộ trước khi ghi đè bằng tile kế tiếp.
- **Biên:** thread không có query ở block cuối vẫn tham gia barrier; tile cuối chỉ đọc số candidate hợp lệ.
- **Giữ nguyên:** XOR/popcount, best/second-best, lowest-index tie-break, ratio test, mutual cross-check và thứ tự matches. Tiling áp dụng cho cả hai chiều matching.
- **Tích hợp:** `run_v2()` gọi `v2_match()`; các stage còn lại dùng hàm V1. CLI chung chọn V0/V1/V2 qua `--version`.

Cell sau trích kernel từ file nguồn thực tế; implementation chỉ được biên dịch từ source, không viết lại trong notebook.

In [ ]:
source = Path('src/stage_implementations/v2_stage4.cu').read_text(encoding='utf-8')
begin = source.index('// All 128 threads')
end = source.index('__global__ void filter_kernel',begin)
display(Code(source[begin:end],language='cuda'))

## d. Result — kiểm chứng, thời gian và speedup

### Kiểm chứng kết quả

Test riêng S4 ở phần setup so V0/V1/V2 trên cùng descriptor đầu vào: số lượng, thứ tự và từng trường `idxA`, `idxB`, `dist`; có các ca input rỗng, ratio boundary, cross-check, tie và biên tile.

Trên mỗi cặp ảnh thật, verifier tái sử dụng extraction V1 để cung cấp cùng descriptor cho ba matcher, sau đó chạy các pipeline để đối chiếu ordered matches, keypoints, inlier, homography và panorama. S1–S3, S5–S6 được kế thừa, không viết lại bộ test nội bộ cho chúng. Verification chạy ngoài các lượt benchmark.

Đề yêu cầu output nhất quán V0 nhưng không ấn định ngưỡng sai số. Tiêu chí nhóm đang dùng ở đây: matches trùng chính xác; inlier trùng; sai số chiếu H trên các điểm và lưới trong ảnh B không quá `1e-3 px`; panorama cùng kích thước, mỗi kênh lệch tối đa 1 mức so với V0, kèm MAE. V1/V2 phải trùng chính xác về matches, phép chiếu và panorama vì chỉ thay cách đọc candidate. Đây là tiêu chí kiểm tra, không phải số liệu đã đo.

Case không tạo được panorama được ghi status/log để phân tích; hai version cùng thất bại không được gọi correctness PASS. Không tính speedup hợp lệ cho kết quả sai hoặc chưa kiểm chứng.

In [ ]:
benchmark_status = run([sys.executable,'src/tools/v2/benchmark.py','--phase','optimized',
                        '--run-dir',RUN_DIR],check=False)
summary = pd.read_csv(RUN_DIR/'summary.csv')
verification_columns = ['case','status','failed_checks','V0_V2_projection_error_px',
    'V0_V2_pixel_max_error','V0_V2_pixel_mae','V1_V2_projection_error_px','V1_V2_pixel_max_error']
display(summary.reindex(columns=verification_columns))
print('Process status:',benchmark_status,'| Reports/logs:',RUN_DIR)

### Thời gian và speedup

Bảng chính tập trung **S4** và **end-to-end**. Mỗi ô thời gian hiển thị **mean (minimum)**, đơn vị ms. Báo cáo đồng thời `T_V0/T_V2` và `T_V1/T_V2`; lớn hơn 1 là nhanh hơn, nhỏ hơn 1 là chậm hơn. S4 GPU là kernel time, còn end-to-end thể hiện lợi ích toàn chương trình.

Timing từng stage, tổng kernel và H2D/D2H vẫn được báo cáo trong bảng chi tiết có thể mở bên dưới và lưu CSV. Không vẽ lại nhiều biểu đồ từ cùng một bộ số liệu. Raw samples và mọi status được giữ trong JSON/log để kiểm tra khi cần.

In [ ]:
valid = summary[summary.status=='pass'].copy()
result_rows = []
for _,row in valid.iterrows():
    entry = {'case':row['case']}
    for metric,label in [('match','S4'),('end_to_end','E2E')]:
        for version in [0,1,2]:
            mean,minimum = row[f'v{version}_{metric}_ms'],row[f'v{version}_{metric}_min_ms']
            entry[f'{label} V{version}: mean (min) ms'] = f'{mean:.3f} ({minimum:.3f})'
        entry[f'{label} V0/V2'] = row[f'v0_over_v2_{metric}_speedup']
        entry[f'{label} V1/V2'] = row[f'v1_over_v2_{metric}_speedup']
    result_rows.append(entry)
display(pd.DataFrame(result_rows))

stage_rows = []
for _,row in summary.iterrows():
    for version in [0,1,2]:
        entry = {'case':row['case'],'version':version,'status':row['status']}
        for metric in ['S1','S2','S3','S4','S5','S6','kernel_total','h2d','d2h','end_to_end']:
            entry[metric+' ms'] = row.get(f'v{version}_{metric}_ms')
        stage_rows.append(entry)
stage_comparison = pd.DataFrame(stage_rows)
stage_comparison.to_csv(RUN_DIR/'stage_comparison.csv',index=False)
display(HTML('<details><summary>Timing chi tiết từng stage, kernel, H2D/D2H và end-to-end (ms)</summary>'
             +stage_comparison.to_html(index=False)+'<p>V0 CPU không có kernel/H2D/D2H; các trường này bằng 0.</p></details>'))

### So sánh speedup trên các thang đo riêng

Một hình gồm bốn ô: **cột trái là V1/V2** để đánh giá trực tiếp tiling, **cột phải là V0/V2** để so với CPU. Hàng trên đo S4, hàng dưới đo end-to-end. Mỗi ô có thang đo riêng để cột V0/V2 lớn không che mức cải thiện V1/V2. Giá trị được ghi ngay trên từng thanh.

Đường đứt tại **1×** là mức bằng nhau: lớn hơn 1 nghĩa là V2 nhanh hơn bản đối chiếu, nhỏ hơn 1 nghĩa là V2 chậm hơn. L1/L2/L3 là ba địa điểm; Challenge là cặp khó. Case không đạt correctness vẫn nằm trong bảng trạng thái, không được vẽ như speedup thành công.


In [ ]:
# V2_RESULT_PLOT: one figure, with independent scales for the two comparisons.
if not valid.empty:
    fig,axes = plt.subplots(2,2,figsize=(16,10))
    indexed = valid.set_index('case')
    def case_label(case):
        return (case.replace('location1_','L1 / ').replace('location2_','L2 / ')
                .replace('location3_','L3 / ').replace('challenge_','Challenge / ')
                .replace('small_720p','720p').replace('medium_1080p','1080p')
                .replace('large_4k','4K'))
    for row,(metric,title) in enumerate([('match','S4 Matching'),('end_to_end','End-to-end')]):
        for column,version,color in [(0,1,'#e68a00'),(1,0,'#2878b5')]:
            ax = axes[row,column]
            values = indexed[f'v{version}_over_v2_{metric}_speedup'].dropna()
            positions = list(range(len(values)))
            ax.barh(positions,values.to_numpy(),color=color,height=0.65)
            ax.set_yticks(positions)
            ax.set_yticklabels([case_label(case) for case in values.index],fontsize=9)
            ax.invert_yaxis()
            ax.axvline(1,color='black',linestyle='--',linewidth=1)
            ax.set_xlim(0,max(1.0,float(values.max()) if len(values) else 1.0)*1.22)
            ax.set_title(f'{title} — V{version}/V2')
            ax.set_xlabel(f'Speedup = time V{version} / time V2')
            ax.grid(axis='x',alpha=0.2)
            ax.set_axisbelow(True)
            for position,value in enumerate(values):
                label = f'{value:.3f}x' if version==1 or value<100 else f'{value:,.0f}x'
                ax.annotate(label,(value,position),xytext=(4,0),textcoords='offset points',
                            va='center',fontsize=9)
    fig.suptitle('V2 speedup — V1 comparison (left), CPU V0 comparison (right)',fontsize=13)
    plt.tight_layout(rect=(0,0,1,0.97)); plt.show()
else:
    print('No verified case to plot; inspect correctness/failure logs.')


### Panorama đại diện và ca khó

Chỉ hiển thị kết quả V2 ở Medium cho từng địa điểm và cặp khó. Ảnh đầy đủ V0/V1/V2 ở mọi độ phân giải vẫn được lưu trong thư mục kết quả. Ảnh xem trước được thu nhỏ để notebook gọn, không thay ảnh dùng kiểm chứng.

Nếu cặp khó thất bại, giữ status và đường dẫn log, giải thích dựa trên ảnh/matches/homography thực tế. Không đoán nguyên nhân từ status, không xem việc hai bản cùng thất bại là panorama đã đúng.

In [ ]:
gallery = summary[summary.case.str.endswith('medium_1080p')]
for _,row in gallery.iterrows():
    display(Markdown(f"**{row['case']} — {row['status']}**"))
    report_path = RUN_DIR/f"{row['case']}_v2.json"
    report = json.loads(report_path.read_text(encoding='utf-8')) if report_path.is_file() else {}
    path = Path(report.get('panorama',{}).get('path',''))
    if report.get('status')=='ok' and path.is_file():
        with PILImage.open(path) as original:
            preview = original.copy()
        preview.thumbnail((800,450))
        display(preview)
    else:
        print('No panorama:',report.get('error','missing output'))
    if row['status']!='pass':
        print('Inspect:',RUN_DIR/f"{row['case']}_verification.log")

## e. Analysis — tối ưu có hiệu quả không?

Đối chiếu kết quả với ý tưởng tái sử dụng candidate:

- Nếu S4 nhanh hơn và end-to-end cũng nhanh hơn, dùng bảng số đo để nêu mức cải thiện theo case/độ phân giải.
- Nếu S4 nhanh hơn nhưng end-to-end thay đổi ít, xem tỷ trọng S4 của V1, copy và các stage còn lại. Không coi các dao động ở stage không đổi là tối ưu mới.
- Nếu S4 chậm hơn, vẫn giữ kết quả. Cache V1, shared-memory access và barrier là các khả năng cần đối chiếu profiler, không phải nguyên nhân đã được chứng minh chỉ bằng thời gian.
- Case correctness không đạt phải được phân tích riêng, không nhận speedup của output sai. Các hạn chế chất lượng/failure case vẫn thuộc đánh giá bài toán panorama.

Cell sau chỉ tóm tắt các con số thực tế. Khi viết báo cáo/slides, bổ sung giải thích gắn với profiler và case cụ thể; có thể chạy profiler V2 riêng khi thật sự cần, không bắt buộc chạy thêm trong luồng chính.

In [ ]:
if not valid.empty:
    faster = valid[valid.v1_over_v2_match_speedup>1]
    slower = valid[valid.v1_over_v2_match_speedup<1]
    display(Markdown(
        f"Có **{len(valid)}/{len(summary)}** case có kết quả được kiểm chứng. "
        f"S4 V2 nhanh hơn V1 ở **{len(faster)}** case và chậm hơn ở **{len(slower)}** case. "
        f"Speedup S4 V1/V2: **{valid.v1_over_v2_match_speedup.min():.3f}–"
        f"{valid.v1_over_v2_match_speedup.max():.3f}×**; end-to-end V1/V2: "
        f"**{valid.v1_over_v2_end_to_end_speedup.min():.3f}–"
        f"{valid.v1_over_v2_end_to_end_speedup.max():.3f}×**."))
    print('Slower S4 cases:',slower.case.tolist())
failed = summary[summary.status!='pass']
print('Cases requiring failure/correctness analysis:',failed.case.tolist())
print('V1 profiling:',profile_v1['status'])
print('Preserve notebook outputs and this result directory:',RUN_DIR)